# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikmannam/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

I chose this lane because it maps to the most direct editorial decision: which pages need a human's attention first. The starter pipeline already demonstrates the workflow (baseline rule, learned model, precision@K), and the starter dataset has the signals needed — impressions, CTR, position, trend direction, engagement — to build a ranked review queue. The question is practical, the output is actionable, and the 7-week arc from baseline to validation fits naturally.

In [1]:
# Setup: imports and data load (used across all cells below)
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent.parent
DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(2)


Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7


## 2. The question: decision, action, cost of a wrong call

**What decision does this improve?**
Which pages across a client's content inventory should an editor review first for possible refresh, rewrite, consolidation, or monitoring.

**Who acts on the output, and what do they do?**
A content editor or SEO specialist with limited weekly capacity (e.g. 20–50 pages per week). They open the ranked queue, read the reason codes for the top candidates, and decide what action to take — update the content, merge pages, or leave it alone.

**What does a wrong answer cost?**
A false positive wastes an editor's finite time on a page that does not need help, pushing a truly declining page further down the queue. A false negative means a page that IS losing traffic goes unaddressed — the loss compounds each week the decline is missed. The asymmetry matters: missing a bad decline costs more than reviewing one extra safe page, so the metric should favor recall at the top of the queue.

**Why does data or ML help at all?**
Decline is driven by many weak signals — position drift, CTR erosion, aging content, intent mismatch — that interact in non-obvious ways. A hand-written rule (like the starter baseline) captures some patterns but misses others. A learned model can combine these signals into a single review priority that adapts to patterns in the data.

In [2]:
# Label: decline rate in the raw data
total = len(df)
declining = (df["trend_direction"] == "down").sum()
up = (df["trend_direction"] == "up").sum()
stable = (df["trend_direction"] == "stable").sum()

print(f"Total content items: {total:,}")
print(f"Declining (trend_direction == 'down'): {declining:,} ({declining/total*100:.1f}%)")
print(f"Up: {up:,} ({up/total*100:.1f}%)")
print(f"Stable: {stable:,} ({stable/total*100:.1f}%)")
print()
print("More than half of all pages show declining trend — the problem is widespread.")

Total content items: 30,000
Declining (trend_direction == 'down'): 16,262 (54.2%)
Up: 4,388 (14.6%)
Stable: 5,962 (19.9%)

More than half of all pages show declining trend — the problem is widespread.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter data that make this lane worth the next 7 weeks:

In [3]:
# 3 real numbers from the starter data

# 1) Decline prevalence
pct_declining = declining / total * 100

# 2) Average CTR (rate columns are x100, so 0.51 means 0.51%)
avg_ctr = df["ctr"].mean()

# 3) Total impressions across all items — signals real demand
total_impressions = df["impressions_90d"].sum()
median_impressions = df["impressions_90d"].median()

print(f"1. {pct_declining:.1f}% of pages have a declining trend direction ({declining:,} of {total:,})")
print(f"2. Average CTR across all pages: {avg_ctr:.2f}% (median CTR: {df['ctr'].median():.2f}%)")
print(f"3. Total impressions (90d): {total_impressions:,} | Median per page: {median_impressions:,}")
print()
print("54.2% declining + low average CTR (0.51%) + substantial demand (156M total impressions)")
print("= a large pool of visible pages that may need attention.")

1. 54.2% of pages have a declining trend direction (16,262 of 30,000)
2. Average CTR across all pages: 0.51% (median CTR: 0.07%)
3. Total impressions (90d): 156,010,989 | Median per page: 731.0

54.2% declining + low average CTR (0.51%) + substantial demand (156M total impressions)
= a large pool of visible pages that may need attention.


## 4. Careful words: what I can and can't claim

**What this work will claim:**
- Observed associations between content/signal features and declining trend direction.
- A ranked queue that prioritizes pages for human review, measured by precision@K.
- Directional comparisons: "pages with these characteristics tend to show decline more often."
- Decision-support results: the queue helps an editor triage, it does not replace them.

**What this work will NOT claim:**
- Causal proof that refreshing a recommended page caused a recovery. The data is observational — no experiment, no counterfactual.
- "Predicting Google" or any search algorithm factor. The data shows correlations from one anonymized snapshot.
- Universal rules that apply to every client or content type. Results are specific to this dataset and validation split.

In [4]:
# Quick sanity: what does the baseline pipeline's Precision@50 look like?
# From the committed outputs (model_report.md):
#   baseline rule Precision@50 = 0.240
#   random forest Precision@50 = 0.740
# This gives a benchmark to beat as I iterate.
print("Starter pipeline benchmark (from committed outputs):")
print("  Baseline rule Precision@50:  0.240 (12/50 correct)")
print("  Random forest Precision@50:  0.740 (37/50 correct)")
print("  Base declining rate in data: ~54% — the baseline already lifts above random.")

Starter pipeline benchmark (from committed outputs):
  Baseline rule Precision@50:  0.240 (12/50 correct)
  Random forest Precision@50:  0.740 (37/50 correct)
  Base declining rate in data: ~54% — the baseline already lifts above random.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.